In [5]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pyref.fitting as fit
import os

In [6]:
from utils.profile_slab import AdaptiveBookendedProfile
from utils.slab_builders import vacuum, sio2, substrate
from utils import read_xrr, read_ooc
from utils.helpers.fitting_helper import package_results


oocs = read_ooc("dft.csv", material="znpc")
data = read_xrr("reflectivity_data", material="znpc")

def model_bookend(
    energy: float,
    num_slabs: int = 40,
    tau_si: float = 8,
    tau_vac: float = 10,
    alpha_bulk: float = np.pi / 2,
    alpha_si: float = 0,
    alpha_vac: float = 0,
    rho_bulk=1.61,
    rho_si=1.61,
    rho_vac=1.61,
    decay_length_substrate=10,
    decay_length_vacuum=10,
    surface_roughness: float = 10,
):
    return (
        vacuum(energy)  # pyright: ignore[reportOperatorIssue]
        | AdaptiveBookendedProfile(
            oocs,
            energy=energy,
            total_thick=200,
            surface_roughness=surface_roughness,
            tau_si=tau_si,
            tau_vac=tau_vac,
            alpha_bulk=alpha_bulk,
            alpha_si=alpha_si,
            alpha_vac=alpha_vac,
            rho_bulk=rho_bulk,
            rho_si=rho_si,
            rho_vac=rho_vac,
            decay_length_substrate=decay_length_substrate,
            decay_length_vacuum=decay_length_vacuum,
            num_slabs=num_slabs,
            name=f"ZnPc_{energy:.1f}",
        )
        | sio2(energy)
        | substrate(energy)
    )

In [7]:
import pyref.fitting as fit
from pyref.fitting import Transform

from utils.slab_builders import select

objective = [
    fit.AnisotropyObjective(
        fit.ReflectModel(model_bookend(250), energy=250),
        data["250.0"],
        logp_anisotropy_weight=0.5,
        transform=Transform("logY"),
    ),
    fit.AnisotropyObjective(
        fit.ReflectModel(model_bookend(275), energy=275),
        data["275.0"],
        logp_anisotropy_weight=0.5,
        transform=Transform("logY"),
    ),
    fit.AnisotropyObjective(
        fit.ReflectModel(model_bookend(283.7), energy=283.7),
        data["283.7"],
        logp_anisotropy_weight=0.5,
        transform=Transform("logY"),
    ),
]


select(objective[0], "Oxide").thick.setp(vary=False)
select(objective[0], "Oxide").rough.setp(vary=False)

select(objective[0], "ZnPc").total_thick.setp(vary=True, bounds=(140, 210))
select(objective[0], "ZnPc").surface_roughness.setp(vary=False, bounds=(0, 50))
select(objective[0], "ZnPc").tau_si.setp(vary=True, bounds=(0, 40))
select(objective[0], "ZnPc").tau_vac.setp(vary=True, bounds=(0, 40))
select(objective[0], "ZnPc").alpha_bulk.setp(vary=True, bounds=(0, np.pi / 2))
select(objective[0], "ZnPc").alpha_si.setp(vary=False, bounds=(0, np.pi / 2))
select(objective[0], "ZnPc").alpha_vac.setp(vary=False, bounds=(0, np.pi / 2))
select(objective[0], "ZnPc").rho_bulk.setp(vary=True, bounds=(1, 2))
select(objective[0], "ZnPc").rho_si.setp(vary=True, bounds=(1, 2))
select(objective[0], "ZnPc").rho_vac.setp(vary=True, bounds=(1, 2))
select(objective[0], "ZnPc").decay_length_substrate.setp(vary=None, constraint=select(objective[0], "ZnPc").tau_si)
select(objective[0], "ZnPc").decay_length_vacuum.setp(vary=None, constraint=select(objective[0], "ZnPc").tau_vac)

# objective[0].model.energy_offset.setp(vary=True, bounds=(-.5, .5))

for i in range(1, len(objective)):
    select(objective[i], "Oxide").thick.setp(constraint=select(objective[0], "Oxide").thick)
    select(objective[i], "Oxide").rough.setp(constraint=select(objective[0], "Oxide").rough)

    select(objective[i], "ZnPc").total_thick.setp(constraint=select(objective[0], "ZnPc").total_thick)
    select(objective[i], "ZnPc").surface_roughness.setp(constraint=select(objective[0], "ZnPc").surface_roughness)
    select(objective[i], "ZnPc").tau_si.setp(constraint=select(objective[0], "ZnPc").tau_si)
    select(objective[i], "ZnPc").tau_vac.setp(constraint=select(objective[0], "ZnPc").tau_vac)
    select(objective[i], "ZnPc").alpha_bulk.setp(constraint=select(objective[0], "ZnPc").alpha_bulk)
    select(objective[i], "ZnPc").alpha_si.setp(constraint=select(objective[0], "ZnPc").alpha_si)
    select(objective[i], "ZnPc").alpha_vac.setp(constraint=select(objective[0], "ZnPc").alpha_vac)
    select(objective[i], "ZnPc").rho_bulk.setp(constraint=select(objective[0], "ZnPc").rho_bulk)
    select(objective[i], "ZnPc").rho_si.setp(constraint=select(objective[0], "ZnPc").rho_si)
    select(objective[i], "ZnPc").rho_vac.setp(constraint=select(objective[0], "ZnPc").rho_vac)
    select(objective[i], "ZnPc").decay_length_substrate.setp(constraint=select(objective[0], "ZnPc").tau_si)
    select(objective[i], "ZnPc").decay_length_vacuum.setp(constraint=select(objective[0], "ZnPc").tau_vac)
    objective[i].model.energy_offset.setp(constraint=objective[0].model.energy_offset)

for o in objective:
    o.model.scale_p.setp(vary=True, bounds=(.5, 2))
    o.model.scale_s.setp(vary=True, bounds=(.5, 2))
    o.model.theta_offset_p.setp(vary=True, bounds=(-.5, .5))
    o.model.theta_offset_s.setp(vary=True, bounds=(-.5, .5))

fitter = fit.CurveFitter(fit.GlobalObjective(objective))
package_results(fitter)

/home/hduva/projects/xrr_notebooks/.venv/lib/python3.12/site-packages/uncertainties/core.py:1024: UserWarning: Using UFloat objects with std_dev==0 may give unexpected results.
  warn("Using UFloat objects with std_dev==0 may give unexpected results.")


,value
name,
scale_s,1.0+/-0
scale_p,1.0+/-0
theta_offset_s,0.0+/-0
theta_offset_p,0.0+/-0
total_thick,200.0+/-0
rho_bulk,1.61+/-0
rho_si,1.61+/-0
rho_vac,1.61+/-0
tau_si,8.0+/-0


In [8]:
fitter = fit.CurveFitter(fit.GlobalObjective(objective))
fitter.fit()
package_results(fitter)

23.88750405965288: : 161it [09:28,  3.53s/it] 
/home/hduva/projects/xrr_notebooks/src/utils/profile_slab.py:1005: RuntimeWarning: invalid value encountered in cos
  c = np.square(np.cos(ori))
/home/hduva/projects/xrr_notebooks/src/utils/profile_slab.py:1006: RuntimeWarning: invalid value encountered in sin
  s = np.square(np.sin(ori))
/home/hduva/projects/xrr_notebooks/.venv/lib/python3.12/site-packages/pyref/fitting/uniaxial.py:221: RuntimeWarning: invalid value encountered in scalar divide
  nu = (e_e - e_o) / e_o  # intermediate birefringence from reference
/home/hduva/projects/xrr_notebooks/.venv/lib/python3.12/site-packages/pyref/fitting/uniaxial.py:242: RuntimeWarning: invalid value encountered in scalar divide
  kz_extraord = (1 / (1 + nu * na**2)) * (
/home/hduva/projects/xrr_notebooks/.venv/lib/python3.12/site-packages/pyref/fitting/uniaxial.py:290: RuntimeWarning: invalid value encountered in scalar divide
  nu = (e_e - e_o) / e_o  # intermediate birefringence from reference


RuntimeError: Objective.logl encountered a NaN.